<a href="https://colab.research.google.com/github/eteitelbaum/code-satp/blob/main/training-and-inference-notebooks/location_inference_googleapi_and_t5_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


Using device: cpu
Using device: cpu


In [ ]:
def extract_locations(summary):
    # Prepare the input text
    input_text = f"Extract the location of the incident: {summary}"
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

# Function to get location details using Google Geocoding API
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, and town/village (if any)."""
    #query = ', '.join(locations)
    params = {
        'address': locations,
        'key': API_KEY
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Parse the address components
    results = data.get('results', [])
    if not results:
        print("No results found.")
        return None

    # Take the first result
    address_components = results[0]['address_components']

    # Initialize components
    state = district = subdistrict = town_village = None

    # Map Google address types to desired components
    for component in address_components:
        types = component['types']
        if 'administrative_area_level_1' in types:
            state = component['long_name']
        elif 'administrative_area_level_2' in types:
            district = component['long_name']
        elif 'administrative_area_level_3' in types:
            subdistrict = component['long_name']
        elif 'locality' in types:
            town_village = component['long_name']
        elif 'sublocality' in types and not town_village:
            town_village = component['long_name']

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village
    }

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"


# Updated function to get location details including latitude and longitude
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, town/village, and latitude/longitude of the most specific level."""
    #query = ', '.join(locations)
    params = {
        'address': locations,
        'key': API_KEY
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Initialize components
    state = district = subdistrict = town_village = None
    latitude = longitude = None
    found_level = None  # Keep track of the most specific level found

    # Desired levels in order of specificity
    levels = ['locality', 'administrative_area_level_3', 'administrative_area_level_2']

    # Iterate over results to find the most specific level
    for result in data.get('results', []):
        temp_state = temp_district = temp_subdistrict = temp_town_village = None
        address_components = result['address_components']

        # Map address components
        for component in address_components:
            types = component['types']
            if 'administrative_area_level_1' in types:
                temp_state = component['long_name']
            elif 'administrative_area_level_2' in types:
                temp_district = component['long_name']
            elif 'administrative_area_level_3' in types:
                temp_subdistrict = component['long_name']
            elif 'locality' in types:
                temp_town_village = component['long_name']
            elif 'sublocality' in types and not temp_town_village:
                temp_town_village = component['long_name']

        # Determine the most specific level in this result
        if temp_town_village and found_level not in ['town_village']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = temp_town_village
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'town_village'
        elif temp_subdistrict and found_level not in ['town_village', 'subdistrict']:
            state = temp_state
            district = temp_district
            subdistrict = temp_subdistrict
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'subdistrict'
        elif temp_district and found_level not in ['town_village', 'subdistrict', 'district']:
            state = temp_state
            district = temp_district
            subdistrict = None
            town_village = None
            location = result['geometry']['location']
            latitude = location['lat']
            longitude = location['lng']
            found_level = 'district'

        # Break the loop if the most specific level is found
        if found_level == 'town_village':
            break

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village,
        'latitude': latitude,
        'longitude': longitude
    }

In [ ]:
incident_summary = 'A Communist Party of India-Maoist (CPI-Maoist) cadre demanded an extortion of INR 1 million from Block Development Officer (BDO), Satish Kumar, of Jhajha in Jamui District on January 20, reports Web India. The sender in the letter identified himself as Umesh Yadav, and claimed himself a native of Jhajha Police Station area. BDO lodged an FIR at Jhajha Police Station.'
locations = extract_locations(incident_summary)
location_details = get_location_details(locations)
#print(location_details)

if location_details:
  location_details.get('state')
  location_details.get('district')
  location_details.get('subdistrict')
  location_details.get('town_village')

{'state': 'Bihar', 'district': 'Munger Division', 'subdistrict': 'Jamui', 'town_village': 'Jhajha', 'latitude': 24.7744999, 'longitude': 86.3756867}


In [ ]:
type(locations)

str

In [ ]:

# Load the incident data
satp_dat = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_id.csv', dtype=str)
satp_data = satp_dat.head(100).copy()

satp_data['extracted_state'] = None
satp_data['extracted_district'] = None
satp_data['extracted_subdistrict'] = None
satp_data['extracted_town_village'] = None


# Process each summary in the data
for index, row in satp_data.iterrows():
    summary = row['incident_summary']  # Adjust column name if necessary
    locations = extract_locations(summary)
    satp_data.at[index, 'extracted_locations'] = locations
    if locations:
        location_details = get_location_details(locations)
        if location_details:
            # Update the DataFrame with location details
            satp_data.at[index, 'extracted_state'] = location_details.get('state')
            satp_data.at[index, 'extracted_district'] = location_details.get('district')
            satp_data.at[index, 'extracted_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'extracted_town_village'] = location_details.get('town_village')
    else:
        print(f"No locations found in summary at index {index}")
